In [16]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED

PROCESSED_DIR = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed")
QUESTIONS_FILE = Path(r"D:\Lokesh files\Projects\LearningDNA\contents\questions.csv")
VECTOR_DIR = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_vectors")

NUM_THREADS = 6
QUESTION_DIM = 64
SKILL_DIM = 32
CORRECTNESS_DIM = 8
ELAPSED_DIM = 16
ATTEMPT_DIM = 8
TIME_DELTA_DIM = 8
INTERACTION_DIM = 136
MODEL_DIM = 128

VECTOR_DIR.mkdir(parents=True, exist_ok=True)

parts = sorted(PROCESSED_DIR.glob("part_*.parquet"))

print("Processed parts:", f"{len(parts):,}")
print("Questions file:", QUESTIONS_FILE)
print("Output directory:", VECTOR_DIR)
print("Threads:", NUM_THREADS)

Processed parts: 15,687
Questions file: D:\Lokesh files\Projects\LearningDNA\contents\questions.csv
Output directory: D:\Lokesh files\Projects\LearningDNA\processed\kt1_vectors
Threads: 6


In [2]:
sample = pd.read_parquet(parts[0])

print("Shape:", sample.shape)
print("\nColumns:")
print(sample.columns.tolist())
print("\nData types:")
print(sample.dtypes)
print("\nSample:")
display(sample.head())

Shape: (41088, 15)

Columns:
['user_id', 'timestamp', 'solving_id', 'question_id', 'user_answer', 'correct_answer', 'correctness', 'answer_missing', 'tags', 'elapsed_time', 'elapsed_seconds', 'attempt_index', 'time_delta_seconds', 'log_elapsed_time', 'log_time_delta']

Data types:
user_id                        int64
timestamp                      int64
solving_id                     int64
question_id           string[python]
user_answer           string[python]
correct_answer        string[python]
correctness                     int8
answer_missing                  int8
tags                  string[python]
elapsed_time                   int64
elapsed_seconds              float64
attempt_index                  int32
time_delta_seconds           float64
log_elapsed_time             float64
log_time_delta               float64
dtype: object

Sample:


,user_id,timestamp,solving_id,question_id,user_answer,correct_answer,correctness,answer_missing,tags,elapsed_time,elapsed_seconds,attempt_index,time_delta_seconds,log_elapsed_time,log_time_delta
0,1,1565096190868,1,q5012,b,c,0,0,74,38000,38.0,1,0.000,3.663562,0.000000
1,1,1565096221062,2,q4706,c,c,1,0,71,24000,24.0,1,30.194,3.218876,3.440226
2,1,1565096293432,3,q4366,b,b,1,0,103,68000,68.0,1,72.370,4.234107,4.295515
3,1,1565096339668,4,q4829,a,c,0,0,83,42000,42.0,1,46.236,3.761200,3.855156
4,1,1565096401774,5,q6528,b,d,0,0,90,59000,59.0,1,62.106,4.094345,4.144816


In [3]:
print("Building question vocabulary...")

question_ids = set()

for i, part in enumerate(parts, 1):
    df = pd.read_parquet(part, columns=["question_id"])
    question_ids.update(df["question_id"].dropna().unique())

    if i % 1000 == 0 or i == len(parts):
        print(f"[{i:,}/{len(parts):,}] Questions collected: {len(question_ids):,}")

question_to_idx = {
    q: i for i, q in enumerate(sorted(question_ids))
}

num_questions = len(question_to_idx)

print("\nUnique questions:", f"{num_questions:,}")
print("Question embedding:", f"{num_questions:,} × {QUESTION_DIM}")

Building question vocabulary...
[1,000/15,687] Questions collected: 12,284
[2,000/15,687] Questions collected: 12,284
[3,000/15,687] Questions collected: 12,284
[4,000/15,687] Questions collected: 12,284
[5,000/15,687] Questions collected: 12,284
[6,000/15,687] Questions collected: 12,284
[7,000/15,687] Questions collected: 12,284
[8,000/15,687] Questions collected: 12,284
[9,000/15,687] Questions collected: 12,284
[10,000/15,687] Questions collected: 12,284
[11,000/15,687] Questions collected: 12,284
[12,000/15,687] Questions collected: 12,284
[13,000/15,687] Questions collected: 12,284
[14,000/15,687] Questions collected: 12,284
[15,000/15,687] Questions collected: 12,284
[15,687/15,687] Questions collected: 12,284

Unique questions: 12,284
Question embedding: 12,284 × 64


In [4]:
print("Building skill vocabulary...")

skill_set = set()

for i, part in enumerate(parts, 1):
    df = pd.read_parquet(part, columns=["tags"])

    for tags in df["tags"].dropna():
        for tag in str(tags).split(";"):
            tag = tag.strip()
            if tag:
                skill_set.add(tag)

    if i % 1000 == 0 or i == len(parts):
        print(f"[{i:,}/{len(parts):,}] Skills collected: {len(skill_set):,}")

skill_to_idx = {
    skill: i + 1 for i, skill in enumerate(sorted(skill_set))
}

skill_to_idx["<PAD>"] = 0

num_skills = len(skill_to_idx)

print("\nUnique skills:", f"{num_skills:,}")
print("Skill embedding:", f"{num_skills:,} × {SKILL_DIM}")

Building skill vocabulary...
[1,000/15,687] Skills collected: 189
[2,000/15,687] Skills collected: 189
[3,000/15,687] Skills collected: 189
[4,000/15,687] Skills collected: 189
[5,000/15,687] Skills collected: 189
[6,000/15,687] Skills collected: 189
[7,000/15,687] Skills collected: 189
[8,000/15,687] Skills collected: 189
[9,000/15,687] Skills collected: 189
[10,000/15,687] Skills collected: 189
[11,000/15,687] Skills collected: 189
[12,000/15,687] Skills collected: 189
[13,000/15,687] Skills collected: 189
[14,000/15,687] Skills collected: 189
[15,000/15,687] Skills collected: 189
[15,687/15,687] Skills collected: 189

Unique skills: 190
Skill embedding: 190 × 32


In [5]:
import pickle

with open(VECTOR_DIR / "question_vocab.pkl", "wb") as f:
    pickle.dump(question_to_idx, f)

with open(VECTOR_DIR / "skill_vocab.pkl", "wb") as f:
    pickle.dump(skill_to_idx, f)

print("✓ Question vocabulary saved.")
print("✓ Skill vocabulary saved.")

✓ Question vocabulary saved.
✓ Skill vocabulary saved.


In [6]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_questions, num_skills):
        super().__init__()

        self.question_embedding = nn.Embedding(
            num_questions,
            QUESTION_DIM
        )

        self.skill_embedding = nn.Embedding(
            num_skills,
            SKILL_DIM,
            padding_idx=0
        )

        self.correctness_embedding = nn.Embedding(
            3,
            CORRECTNESS_DIM
        )

        self.elapsed_projection = nn.Linear(
            1,
            ELAPSED_DIM
        )

        self.attempt_embedding = nn.Embedding(
            256,
            ATTEMPT_DIM
        )

        self.time_delta_projection = nn.Linear(
            1,
            TIME_DELTA_DIM
        )

        self.interaction_projection = nn.Linear(
            INTERACTION_DIM,
            MODEL_DIM
        )

    def forward(
        self,
        question_idx,
        skill_idx,
        correctness_idx,
        elapsed,
        attempt_idx,
        time_delta
    ):
        q = self.question_embedding(question_idx)
        s = self.skill_embedding(skill_idx).mean(dim=1)
        c = self.correctness_embedding(correctness_idx)
        e = self.elapsed_projection(elapsed)
        a = self.attempt_embedding(attempt_idx)
        t = self.time_delta_projection(time_delta)

        combined = torch.cat([q, s, c, e, a, t], dim=1)

        return self.interaction_projection(combined)

In [7]:
encoder = FeatureEncoder(
    num_questions=num_questions,
    num_skills=num_skills
)

encoder.eval()

print(encoder)

FeatureEncoder(
  (question_embedding): Embedding(12284, 64)
  (skill_embedding): Embedding(190, 32, padding_idx=0)
  (correctness_embedding): Embedding(3, 8)
  (elapsed_projection): Linear(in_features=1, out_features=16, bias=True)
  (attempt_embedding): Embedding(256, 8)
  (time_delta_projection): Linear(in_features=1, out_features=8, bias=True)
  (interaction_projection): Linear(in_features=136, out_features=128, bias=True)
)


In [8]:
def prepare_features(df):
    question_idx = torch.tensor(
        df["question_id"].map(question_to_idx).fillna(0).to_numpy(),
        dtype=torch.long
    )

    skill_indices = []
    max_tags = 1

    for tags in df["tags"].fillna(""):
        current = [
            skill_to_idx.get(tag.strip(), 0)
            for tag in str(tags).split(";")
            if tag.strip()
        ]

        if not current:
            current = [0]

        skill_indices.append(current)
        max_tags = max(max_tags, len(current))

    skill_tensor = torch.zeros(
        len(df),
        max_tags,
        dtype=torch.long
    )

    for i, skills in enumerate(skill_indices):
        skill_tensor[i, :len(skills)] = torch.tensor(
            skills,
            dtype=torch.long
        )

    correctness_idx = torch.tensor(
        np.where(
            df["answer_missing"].to_numpy() == 1,
            2,
            df["correctness"].to_numpy()
        ),
        dtype=torch.long
    )

    elapsed = torch.tensor(
        df["log_elapsed_time"].to_numpy(),
        dtype=torch.float32
    ).unsqueeze(1)

    attempt_idx = torch.tensor(
        np.clip(
            df["attempt_index"].to_numpy() - 1,
            0,
            255
        ),
        dtype=torch.long
    )

    time_delta = torch.tensor(
        df["log_time_delta"].to_numpy(),
        dtype=torch.float32
    ).unsqueeze(1)

    return (
        question_idx,
        skill_tensor,
        correctness_idx,
        elapsed,
        attempt_idx,
        time_delta
    )

In [9]:
test_df = pd.read_parquet(parts[0])

features = prepare_features(test_df)

for name, tensor in zip(
    [
        "question",
        "skills",
        "correctness",
        "elapsed",
        "attempt",
        "time_delta"
    ],
    features
):
    print(f"{name:12} {tuple(tensor.shape)}")

question     (41088,)
skills       (41088, 7)
correctness  (41088,)
elapsed      (41088, 1)
attempt      (41088,)
time_delta   (41088, 1)


In [10]:
with torch.no_grad():
    test_vectors = encoder(*features)

print("\nInput interactions:", len(test_df))
print("Output shape:", tuple(test_vectors.shape))

assert test_vectors.shape == (
    len(test_df),
    MODEL_DIM
)

assert torch.isfinite(test_vectors).all()

print("✓ 136D features → 128D vectors")


Input interactions: 41088
Output shape: (41088, 128)
✓ 136D features → 128D vectors


In [11]:
print("Question ID:", test_df.iloc[0]["question_id"])
print("Correctness:", test_df.iloc[0]["correctness"])
print("Answer missing:", test_df.iloc[0]["answer_missing"])
print("Attempt:", test_df.iloc[0]["attempt_index"])

print("\n128D vector:")
print(test_vectors[0])

print("\nVector norm:", test_vectors[0].norm().item())

Question ID: q5012
Correctness: 0
Answer missing: 0
Attempt: 1

128D vector:
tensor([-0.2728, -0.2865, -0.2477,  0.5616, -0.8683, -1.0469,  0.5192, -1.0723,
         1.1534, -0.5618,  0.2188, -1.0498,  0.2917, -0.3448, -0.3541, -0.3848,
         1.0281, -0.7964, -0.0196,  0.5307,  0.6535, -0.2010,  0.5739,  1.3431,
         0.1294, -0.1389,  0.1841,  0.1391, -0.3041, -0.3301,  0.3659,  0.2221,
         0.2518,  0.8361, -0.1597,  0.7758, -1.1510, -0.1624, -0.5708,  0.4532,
        -0.3478, -0.4380,  0.2938, -0.8990,  0.4869, -0.3591,  0.5569,  0.2744,
        -0.1400,  0.5731, -0.0891, -1.1146,  0.0017,  0.8000,  0.6976, -0.5872,
         0.3991, -0.4542,  0.9030, -0.5979, -0.3698,  0.2752,  1.1285,  0.2398,
         0.0485,  0.4335, -0.2036, -0.3539,  0.8016,  0.6051,  0.3511,  0.2186,
         0.1816, -0.2636,  0.5640, -0.8687,  0.0981,  0.3675,  0.3926, -0.9262,
        -0.2393,  1.0440, -0.0562, -0.1563,  0.2527, -0.9210, -0.3931,  0.6580,
        -0.2628, -0.6968, -0.4716,  1.4536,

In [12]:
test_output = VECTOR_DIR / "test_part_000000.pt"

torch.save(
    {
        "user_id": torch.tensor(
            test_df["user_id"].to_numpy(),
            dtype=torch.long
        ),
        "timestamp": torch.tensor(
            test_df["timestamp"].to_numpy(),
            dtype=torch.long
        ),
        "question_id": test_df["question_id"].astype(str).tolist(),
        "vectors": test_vectors
    },
    test_output
)

print("Saved:", test_output)

Saved: D:\Lokesh files\Projects\LearningDNA\processed\kt1_vectors\test_part_000000.pt


In [17]:
print("Loading question metadata...")

questions_df = pd.read_csv(
    QUESTIONS_FILE,
    dtype={
        "question_id": "string",
        "correct_answer": "string",
        "tags": "string"
    },
    usecols=["question_id", "correct_answer", "tags"]
)

questions_df = questions_df.drop_duplicates(
    subset=["question_id"]
).reset_index(drop=True)

print("Questions loaded:", f"{len(questions_df):,}")

print("Building fast question → skill mapping...")

question_skill_map = {}

for question_id, tags in zip(
    questions_df["question_id"],
    questions_df["tags"]
):
    skills = [
        skill_to_idx.get(tag.strip(), 0)
        for tag in str(tags).split(";")
        if tag.strip()
    ]

    question_skill_map[str(question_id)] = skills or [0]

max_skills = max(len(v) for v in question_skill_map.values())

question_skill_tensor = torch.zeros(
    len(question_to_idx),
    max_skills,
    dtype=torch.long
)

for qid, qidx in question_to_idx.items():
    skills = question_skill_map.get(str(qid), [0])
    question_skill_tensor[qidx, :len(skills)] = torch.tensor(
        skills,
        dtype=torch.long
    )

print("Questions:", f"{len(question_to_idx):,}")
print("Maximum skills per question:", max_skills)
print("Question skill tensor:", tuple(question_skill_tensor.shape))

Loading question metadata...
Questions loaded: 13,169
Building fast question → skill mapping...
Questions: 12,284
Maximum skills per question: 7
Question skill tensor: (12284, 7)


In [18]:
def prepare_features_fast(df):
    question_idx = torch.from_numpy(
        df["question_id"].map(question_to_idx).fillna(0).to_numpy(dtype=np.int64)
    )

    skill_tensor = question_skill_tensor[question_idx]

    correctness_idx = torch.from_numpy(
        np.where(
            df["answer_missing"].to_numpy() == 1,
            2,
            df["correctness"].to_numpy()
        ).astype(np.int64)
    )

    elapsed = torch.from_numpy(
        df["log_elapsed_time"].to_numpy(dtype=np.float32)
    ).unsqueeze(1)

    attempt_idx = torch.from_numpy(
        np.clip(
            df["attempt_index"].to_numpy(dtype=np.int64) - 1,
            0,
            255
        )
    )

    time_delta = torch.from_numpy(
        df["log_time_delta"].to_numpy(dtype=np.float32)
    ).unsqueeze(1)

    return (
        question_idx,
        skill_tensor,
        correctness_idx,
        elapsed,
        attempt_idx,
        time_delta
    )

In [19]:
import time

benchmark_part = parts[100]

start = time.time()
df = pd.read_parquet(benchmark_part)
read_time = time.time() - start

start = time.time()
features = prepare_features_fast(df)
prep_time = time.time() - start

start = time.time()
with torch.inference_mode():
    vectors = encoder(*features)
encode_time = time.time() - start

total_time = read_time + prep_time + encode_time

print(f"Rows: {len(df):,}")
print(f"Read: {read_time:.3f}s")
print(f"Feature preparation: {prep_time:.3f}s")
print(f"Encoding: {encode_time:.3f}s")
print(f"Total: {total_time:.3f}s")
print(f"Rate: {len(df) / total_time:,.0f} rows/s")

Rows: 40,262
Read: 0.219s
Feature preparation: 0.156s
Encoding: 0.114s
Total: 0.489s
Rate: 82,362 rows/s


In [20]:
def process_vector_part(input_file, output_file):
    try:
        if output_file.exists():
            return {
                "file": input_file.name,
                "rows": 0,
                "status": "skipped",
                "error": None
            }

        df = pd.read_parquet(
            input_file,
            columns=[
                "user_id",
                "timestamp",
                "solving_id",
                "question_id",
                "correctness",
                "answer_missing",
                "log_elapsed_time",
                "attempt_index",
                "log_time_delta"
            ]
        )

        features = prepare_features_fast(df)

        with torch.inference_mode():
            vectors = encoder(*features)

        torch.save(
            {
                "user_id": torch.from_numpy(df["user_id"].to_numpy(dtype=np.int64)),
                "timestamp": torch.from_numpy(df["timestamp"].to_numpy(dtype=np.int64)),
                "solving_id": torch.from_numpy(df["solving_id"].to_numpy(dtype=np.int64)),
                "question_idx": features[0],
                "vectors": vectors
            },
            output_file
        )

        return {
            "file": input_file.name,
            "rows": len(df),
            "status": "success",
            "error": None
        }

    except Exception as e:
        return {
            "file": input_file.name,
            "rows": 0,
            "status": "failed",
            "error": str(e)
        }

In [21]:
import os
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

print("CPU cores:", os.cpu_count())
print("PyTorch threads:", torch.get_num_threads())
print("PyTorch interop threads:", torch.get_num_interop_threads())

CPU cores: 8
PyTorch threads: 1
PyTorch interop threads: 1


In [22]:
NUM_THREADS = 6
MAX_IN_FLIGHT = NUM_THREADS * 2

print("Starting optimized feature-vector generation...")

start = time.time()
completed = 0
total_rows = 0
skipped = 0
failed = []

part_iter = iter(parts)
futures = {}

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    for _ in range(MAX_IN_FLIGHT):
        part = next(part_iter, None)
        if part is None:
            break

        output_file = VECTOR_DIR / part.with_suffix(".pt").name

        futures[executor.submit(
            process_vector_part,
            part,
            output_file
        )] = part.name

    while futures:
        done, _ = wait(
            futures,
            return_when=FIRST_COMPLETED
        )

        for future in done:
            result = future.result()
            del futures[future]

            completed += 1

            if result["status"] == "success":
                total_rows += result["rows"]
            elif result["status"] == "skipped":
                skipped += 1
            else:
                failed.append(result)

            part = next(part_iter, None)

            if part is not None:
                output_file = VECTOR_DIR / part.with_suffix(".pt").name

                futures[executor.submit(
                    process_vector_part,
                    part,
                    output_file
                )] = part.name

            if completed % 100 == 0 or completed == len(parts):
                elapsed = time.time() - start
                rate = total_rows / elapsed if elapsed else 0
                remaining = (
                    (95_293_926 - total_rows) / rate
                    if rate > 0 else 0
                )

                print(
                    f"[{completed:,}/{len(parts):,}] "
                    f"{completed / len(parts) * 100:6.2f}% | "
                    f"Rows: {total_rows:,} | "
                    f"Rate: {rate:,.0f} rows/s | "
                    f"ETA: {remaining / 60:.1f} min | "
                    f"Skipped: {skipped:,} | "
                    f"Failed: {len(failed):,}"
                )

elapsed = time.time() - start

print("\nFeature-vector generation complete.")
print("Parts:", f"{completed:,}")
print("Rows:", f"{total_rows:,}")
print("Skipped:", f"{skipped:,}")
print("Failed:", f"{len(failed):,}")
print("Time:", f"{elapsed / 60:.2f} min")

if total_rows:
    print("Average rate:", f"{total_rows / elapsed:,.0f} rows/s")

if failed:
    with open(
        VECTOR_DIR / "failed_parts.txt",
        "w",
        encoding="utf-8"
    ) as f:
        for item in failed:
            f.write(
                f"{item['file']}\t{item['error']}\n"
            )

Starting optimized feature-vector generation...
[100/15,687]   0.64% | Rows: 0 | Rate: 0 rows/s | ETA: 0.0 min | Skipped: 100 | Failed: 0
[200/15,687]   1.27% | Rows: 0 | Rate: 0 rows/s | ETA: 0.0 min | Skipped: 200 | Failed: 0
[300/15,687]   1.91% | Rows: 0 | Rate: 0 rows/s | ETA: 0.0 min | Skipped: 300 | Failed: 0
[400/15,687]   2.55% | Rows: 0 | Rate: 0 rows/s | ETA: 0.0 min | Skipped: 400 | Failed: 0
[500/15,687]   3.19% | Rows: 0 | Rate: 0 rows/s | ETA: 0.0 min | Skipped: 500 | Failed: 0
[600/15,687]   3.82% | Rows: 1,296,948 | Rate: 214,182 rows/s | ETA: 7.3 min | Skipped: 516 | Failed: 0
[700/15,687]   4.46% | Rows: 2,612,224 | Rate: 154,558 rows/s | ETA: 10.0 min | Skipped: 516 | Failed: 0
[800/15,687]   5.10% | Rows: 3,912,057 | Rate: 129,281 rows/s | ETA: 11.8 min | Skipped: 516 | Failed: 0
[900/15,687]   5.74% | Rows: 5,210,455 | Rate: 134,494 rows/s | ETA: 11.2 min | Skipped: 516 | Failed: 0
[1,000/15,687]   6.37% | Rows: 6,003,471 | Rate: 124,499 rows/s | ETA: 12.0 min | S

In [23]:
print("Validating complete vector dataset...")

vector_parts = sorted(VECTOR_DIR.glob("part_*.pt"))

print("Vector parts found:", f"{len(vector_parts):,}")
print("Expected parts:", f"{len(parts):,}")

total_vectors = 0
unique_users = set()
bad_parts = []

start = time.time()

for i, part in enumerate(vector_parts, 1):
    try:
        data = torch.load(part, map_location="cpu")

        vectors = data["vectors"]
        user_ids = data["user_id"]

        if vectors.ndim != 2 or vectors.shape[1] != 128:
            bad_parts.append((part.name, f"Invalid shape: {vectors.shape}"))

        if len(vectors) != len(user_ids):
            bad_parts.append((part.name, "Vector/user count mismatch"))

        if not torch.isfinite(vectors).all():
            bad_parts.append((part.name, "NaN/Inf detected"))

        total_vectors += len(vectors)
        unique_users.update(user_ids.numpy().tolist())

    except Exception as e:
        bad_parts.append((part.name, str(e)))

    if i % 1000 == 0 or i == len(vector_parts):
        print(
            f"[{i:,}/{len(vector_parts):,}] "
            f"Vectors: {total_vectors:,} | "
            f"Time: {(time.time() - start) / 60:.1f} min"
        )

print("\nFINAL VECTOR VALIDATION")
print("Parts found     :", f"{len(vector_parts):,}")
print("Expected parts  :", f"{len(parts):,}")
print("Total vectors   :", f"{total_vectors:,}")
print("Expected vectors:", f"{95_293_926:,}")
print("Unique students :", f"{len(unique_users):,}")
print("Expected students:", f"{784_309:,}")
print("Bad parts       :", f"{len(bad_parts):,}")

if total_vectors == 95_293_926:
    print("✓ Interaction count PASS")

if len(unique_users) == 784_309:
    print("✓ Student count PASS")

if len(vector_parts) == len(parts):
    print("✓ Part count PASS")

if not bad_parts:
    print("✓ Vector integrity PASS")
else:
    print("\nProblems:")
    for item in bad_parts[:20]:
        print(item)

Validating complete vector dataset...
Vector parts found: 15,687
Expected parts: 15,687
[1,000/15,687] Vectors: 27,785,205 | Time: 6.6 min
[2,000/15,687] Vectors: 36,511,370 | Time: 8.5 min
[3,000/15,687] Vectors: 45,333,514 | Time: 10.4 min
[4,000/15,687] Vectors: 53,442,274 | Time: 12.2 min
[5,000/15,687] Vectors: 60,931,258 | Time: 14.0 min
[6,000/15,687] Vectors: 64,893,773 | Time: 15.3 min
[7,000/15,687] Vectors: 67,713,234 | Time: 16.5 min
[8,000/15,687] Vectors: 71,473,834 | Time: 17.8 min
[9,000/15,687] Vectors: 74,739,866 | Time: 19.1 min
[10,000/15,687] Vectors: 77,899,727 | Time: 20.3 min
[11,000/15,687] Vectors: 81,137,416 | Time: 21.6 min
[12,000/15,687] Vectors: 84,308,043 | Time: 22.8 min
[13,000/15,687] Vectors: 86,964,648 | Time: 24.0 min
[14,000/15,687] Vectors: 89,501,116 | Time: 25.1 min
[15,000/15,687] Vectors: 93,491,296 | Time: 26.4 min
[15,687/15,687] Vectors: 95,293,926 | Time: 27.2 min

FINAL VECTOR VALIDATION
Parts found     : 15,687
Expected parts  : 15,687
